# V18-v2 Figure Merging Notebook
## 26Mar15 — Final Figure Assembly

**Tasks:**
1. **Figure 1** (6 panels): Old F1(A-C) + Old F6(A-C) → New Fig 1(A-F)
2. **Figure 3** (6 panels): Old F3(A,B,D) + Old F4(A,C,D) → New Fig 3(A-F)
3. **Figure 5** (4 panels): Old F8 BCR/TCR — remove title, keep A-D panels only

**Requirements:** PIL (Pillow), matplotlib

**Input:** Individual panel PNG files from Google Drive
**Output:** Merged figures saved to `version18-analysis-v2/figures/`

In [ ]:
# Cell 1: Setup and path configuration
import os
from PIL import Image, ImageDraw, ImageFont
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.gridspec import GridSpec
import numpy as np

# === CONFIGURE PATHS ===
# Change these to match your actual file locations
DRIVE_BASE = '/content/drive/MyDrive/ITLAS'
FIG_DIR = os.path.join(DRIVE_BASE, 'results/version18-analysis-v2/figures')
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'results/version18-analysis-v2/figures/merged_final')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === INPUT FILE PATHS ===
# Update these paths to your actual panel image files
# If panels are in separate files:
PANELS = {
    # Old Figure 1 panels
    'F1A': os.path.join(FIG_DIR, 'Figure1A_liver_bar.png'),
    'F1B': os.path.join(FIG_DIR, 'Figure1B_blood_bar.png'),
    'F1C': os.path.join(FIG_DIR, 'Figure1C_NK_divergence.png'),
    # Old Figure 6 panels
    'F6A': os.path.join(FIG_DIR, 'Figure6A_TGFB1_NK.png'),
    'F6B': os.path.join(FIG_DIR, 'Figure6B_GNLY_CD8T.png'),
    'F6C': os.path.join(FIG_DIR, 'Figure6C_MEFV_CD8T.png'),
    # Old Figure 3 panels (selected)
    'F3A': os.path.join(FIG_DIR, 'Figure3A_HLA-DPB1.png'),
    'F3B': os.path.join(FIG_DIR, 'Figure3B_DNMT1.png'),
    'F3D': os.path.join(FIG_DIR, 'Figure3D_PRDM1.png'),
    # Old Figure 4 panels (selected)
    'F4A': os.path.join(FIG_DIR, 'Figure4A_TOX.png'),
    'F4C': os.path.join(FIG_DIR, 'Figure4C_SOCS1.png'),
    'F4D': os.path.join(FIG_DIR, 'Figure4D_TYROBP.png'),
    # Old Figure 8 (full image)
    'F8_full': os.path.join(FIG_DIR, 'Figure8_BCR_TCR.png'),
}

# === OR if figures are full composite images (not individual panels) ===
# Set these if you have the full figure images and need to crop panels
FULL_FIGURES = {
    'F1_full': os.path.join(FIG_DIR, 'Figure1.png'),
    'F6_full': os.path.join(FIG_DIR, 'Figure6.png'),
    'F3_full': os.path.join(FIG_DIR, 'Figure3.png'),
    'F4_full': os.path.join(FIG_DIR, 'Figure4.png'),
    'F8_full': os.path.join(FIG_DIR, 'Figure8_BCR_TCR.png'),
}

# Check what exists
print('=== Checking panel files ===')
for key, path in PANELS.items():
    exists = os.path.exists(path)
    print(f'  {key}: {"✅" if exists else "❌"} {path}')

print('\n=== Checking full figure files ===')
for key, path in FULL_FIGURES.items():
    exists = os.path.exists(path)
    print(f'  {key}: {"✅" if exists else "❌"} {path}')

print(f'\nOutput directory: {OUTPUT_DIR}')

In [ ]:
# Cell 2: Helper functions for figure merging

def load_and_resize(path, target_height=None, target_width=None):
    """Load image and optionally resize maintaining aspect ratio."""
    img = Image.open(path).convert('RGB')
    if target_height:
        ratio = target_height / img.height
        img = img.resize((int(img.width * ratio), target_height), Image.LANCZOS)
    elif target_width:
        ratio = target_width / img.width
        img = img.resize((target_width, int(img.height * ratio)), Image.LANCZOS)
    return img

def merge_2x3(panels_dict, keys, output_path, title=None, 
              target_panel_width=800, gap=20, label_size=36):
    """
    Merge 6 panels in 2 rows x 3 columns layout.
    keys = list of 6 keys in order [A, B, C, D, E, F]
    """
    labels = ['A', 'B', 'C', 'D', 'E', 'F']
    
    # Load and resize all panels to same width
    imgs = []
    for key in keys:
        path = panels_dict[key]
        if not os.path.exists(path):
            print(f'  ❌ Missing: {key} → {path}')
            return None
        img = load_and_resize(path, target_width=target_panel_width)
        imgs.append(img)
    
    # Calculate layout dimensions
    row1_height = max(imgs[0].height, imgs[1].height, imgs[2].height)
    row2_height = max(imgs[3].height, imgs[4].height, imgs[5].height)
    
    total_width = target_panel_width * 3 + gap * 4  # 3 panels + 4 gaps
    title_height = 60 if title else 0
    total_height = title_height + row1_height + row2_height + gap * 3
    
    # Create canvas
    canvas = Image.new('RGB', (total_width, total_height), 'white')
    draw = ImageDraw.Draw(canvas)
    
    # Try to load a font
    try:
        font_title = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 28)
        font_label = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', label_size)
    except:
        font_title = ImageFont.load_default()
        font_label = ImageFont.load_default()
    
    # Draw title
    if title:
        draw.text((gap, 10), title, fill='black', font=font_title)
    
    # Place panels
    positions = [
        (gap, title_height + gap),  # A
        (gap + target_panel_width + gap, title_height + gap),  # B
        (gap + (target_panel_width + gap) * 2, title_height + gap),  # C
        (gap, title_height + gap + row1_height + gap),  # D
        (gap + target_panel_width + gap, title_height + gap + row1_height + gap),  # E
        (gap + (target_panel_width + gap) * 2, title_height + gap + row1_height + gap),  # F
    ]
    
    for i, (img, pos) in enumerate(zip(imgs, positions)):
        canvas.paste(img, pos)
    
    canvas.save(output_path, dpi=(300, 300))
    print(f'  ✅ Saved: {output_path} ({total_width}x{total_height}px)')
    return canvas

def merge_2x2(panels_dict, keys, output_path, title=None,
              target_panel_width=1000, gap=20):
    """
    Merge 4 panels in 2x2 layout.
    keys = list of 4 keys [A, B, C, D]
    """
    imgs = []
    for key in keys:
        path = panels_dict[key]
        if not os.path.exists(path):
            print(f'  ❌ Missing: {key} → {path}')
            return None
        img = load_and_resize(path, target_width=target_panel_width)
        imgs.append(img)
    
    row1_height = max(imgs[0].height, imgs[1].height)
    row2_height = max(imgs[2].height, imgs[3].height)
    
    total_width = target_panel_width * 2 + gap * 3
    title_height = 60 if title else 0
    total_height = title_height + row1_height + row2_height + gap * 3
    
    canvas = Image.new('RGB', (total_width, total_height), 'white')
    draw = ImageDraw.Draw(canvas)
    
    positions = [
        (gap, title_height + gap),
        (gap + target_panel_width + gap, title_height + gap),
        (gap, title_height + gap + row1_height + gap),
        (gap + target_panel_width + gap, title_height + gap + row1_height + gap),
    ]
    
    for img, pos in zip(imgs, positions):
        canvas.paste(img, pos)
    
    canvas.save(output_path, dpi=(300, 300))
    print(f'  ✅ Saved: {output_path} ({total_width}x{total_height}px)')
    return canvas

def crop_title_from_image(input_path, output_path, title_crop_pixels=80):
    """
    Remove the top N pixels (title area) from an image.
    Adjust title_crop_pixels based on actual title height.
    """
    img = Image.open(input_path).convert('RGB')
    w, h = img.size
    cropped = img.crop((0, title_crop_pixels, w, h))
    cropped.save(output_path, dpi=(300, 300))
    print(f'  ✅ Cropped top {title_crop_pixels}px: {output_path} ({w}x{h-title_crop_pixels}px)')
    return cropped

print('✅ Helper functions loaded')

In [ ]:
# Cell 3: FIGURE 1 — Merge Old F1(A-C) + Old F6(A-C)
# "Immune Cell Composition and Disease Trajectory Across the HBV Spectrum"
# Layout: 2 rows × 3 columns
#   Row 1: A (Liver bar), B (Blood bar), C (NK divergence)
#   Row 2: D (TGFB1 NK), E (GNLY CD8T), F (MEFV CD8T)

print('=== FIGURE 1: Merging F1 + F6 ===')

# Method A: If individual panel files exist
fig1_keys = ['F1A', 'F1B', 'F1C', 'F6A', 'F6B', 'F6C']
all_exist = all(os.path.exists(PANELS[k]) for k in fig1_keys)

if all_exist:
    print('Using individual panel files...')
    merge_2x3(
        PANELS, fig1_keys,
        os.path.join(OUTPUT_DIR, 'Figure1_merged.png'),
        target_panel_width=900
    )
else:
    # Method B: If only full figure images exist, use matplotlib subplot
    f1_path = FULL_FIGURES.get('F1_full', '')
    f6_path = FULL_FIGURES.get('F6_full', '')
    
    if os.path.exists(f1_path) and os.path.exists(f6_path):
        print('Using full figure images (stacking vertically)...')
        img1 = Image.open(f1_path).convert('RGB')
        img6 = Image.open(f6_path).convert('RGB')
        
        # Resize to same width
        target_w = max(img1.width, img6.width)
        if img1.width != target_w:
            ratio = target_w / img1.width
            img1 = img1.resize((target_w, int(img1.height * ratio)), Image.LANCZOS)
        if img6.width != target_w:
            ratio = target_w / img6.width
            img6 = img6.resize((target_w, int(img6.height * ratio)), Image.LANCZOS)
        
        # Stack vertically with small gap
        gap = 30
        total_h = img1.height + img6.height + gap
        canvas = Image.new('RGB', (target_w, total_h), 'white')
        canvas.paste(img1, (0, 0))
        canvas.paste(img6, (0, img1.height + gap))
        
        out_path = os.path.join(OUTPUT_DIR, 'Figure1_merged.png')
        canvas.save(out_path, dpi=(300, 300))
        print(f'  ✅ Saved: {out_path} ({target_w}x{total_h}px)')
    else:
        print('❌ Neither individual panels nor full figures found.')
        print('   Please update PANELS or FULL_FIGURES paths in Cell 1.')
        print(f'   Checked: {f1_path}')
        print(f'   Checked: {f6_path}')

In [ ]:
# Cell 4: FIGURE 3 — Merge selected panels from Old F3 + Old F4
# "IT-Specific Gene Expression Signatures Across Tissue Compartments"
# Layout: 2 rows × 3 columns
#   Row 1: A (HLA-DPB1), B (DNMT1), C (PRDM1)
#   Row 2: D (TOX), E (SOCS1), F (TYROBP)

print('=== FIGURE 3: Merging F3(A,B,D) + F4(A,C,D) ===')

fig3_keys = ['F3A', 'F3B', 'F3D', 'F4A', 'F4C', 'F4D']
all_exist = all(os.path.exists(PANELS[k]) for k in fig3_keys)

if all_exist:
    print('Using individual panel files...')
    merge_2x3(
        PANELS, fig3_keys,
        os.path.join(OUTPUT_DIR, 'Figure3_merged.png'),
        target_panel_width=900
    )
else:
    # Method B: Crop from full figures
    f3_path = FULL_FIGURES.get('F3_full', '')
    f4_path = FULL_FIGURES.get('F4_full', '')
    
    if os.path.exists(f3_path) and os.path.exists(f4_path):
        print('Full figures found. Need to extract specific panels.')
        print('Opening figures to check dimensions...')
        img3 = Image.open(f3_path)
        img4 = Image.open(f4_path)
        print(f'  F3: {img3.size}')
        print(f'  F4: {img4.size}')
        print()
        print('MANUAL STEP NEEDED:')
        print('  Since F3 has 4 panels (A-D) and F4 has 4 panels (A-D),')
        print('  you need to crop individual panels from each figure.')
        print('  Option 1: Use PowerPoint to extract individual panels')
        print('  Option 2: Define crop coordinates below:')
        print()
        print('  # Example crop coordinates (adjust to your images):')
        print('  # F3 is 2x2: A=top-left, B=top-right, C=bottom-left, D=bottom-right')
        print('  # F4 is 2x2: A=top-left, B=top-right, C=bottom-left, D=bottom-right')
        print()
        
        # === UNCOMMENT AND ADJUST THESE COORDINATES ===
        # w3, h3 = img3.size
        # mid_w3, mid_h3 = w3 // 2, h3 // 2
        # F3A = img3.crop((0, 0, mid_w3, mid_h3))          # top-left
        # F3B = img3.crop((mid_w3, 0, w3, mid_h3))          # top-right
        # F3D = img3.crop((mid_w3, mid_h3, w3, h3))          # bottom-right (PRDM1)
        # 
        # w4, h4 = img4.size
        # mid_w4, mid_h4 = w4 // 2, h4 // 2
        # F4A = img4.crop((0, 0, mid_w4, mid_h4))          # top-left (TOX)
        # F4C = img4.crop((0, mid_h4, mid_w4, h4))          # bottom-left (SOCS1)
        # F4D = img4.crop((mid_w4, mid_h4, w4, h4))          # bottom-right (TYROBP)
        #
        # panels = [F3A, F3B, F3D, F4A, F4C, F4D]
        # ... then merge
        
    else:
        print('❌ Panel files and full figures not found.')
        print('   Update paths in Cell 1.')

In [ ]:
# Cell 5: FIGURE 5 — Remove title from BCR/TCR figure
# Old Figure 8 → New Figure 5
# Remove "Figure 8. BCR/TCR Repertoire Analysis: IT-Oriented Key Findings" title
# Keep only the 4 panels (A-D) with their individual subtitles

print('=== FIGURE 5: Remove title from BCR/TCR figure ===')

f8_path = FULL_FIGURES.get('F8_full', PANELS.get('F8_full', ''))

if os.path.exists(f8_path):
    img = Image.open(f8_path).convert('RGB')
    w, h = img.size
    print(f'  Original size: {w}x{h}px')
    
    # Preview top portion to find title height
    # Display top 150 pixels to check where title ends
    preview = img.crop((0, 0, w, min(200, h)))
    plt.figure(figsize=(15, 2))
    plt.imshow(preview)
    plt.title('Preview: Top 200px — find where title ends')
    plt.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='50px')
    plt.axhline(y=80, color='g', linestyle='--', alpha=0.5, label='80px')
    plt.axhline(y=100, color='b', linestyle='--', alpha=0.5, label='100px')
    plt.legend()
    plt.show()
    
    # === ADJUST THIS VALUE based on preview ===
    TITLE_CROP_PIXELS = 80  # Crop top 80 pixels (adjust after preview)
    
    crop_title_from_image(
        f8_path,
        os.path.join(OUTPUT_DIR, 'Figure5_BCR_TCR_notitle.png'),
        title_crop_pixels=TITLE_CROP_PIXELS
    )
    
    # Show result
    result = Image.open(os.path.join(OUTPUT_DIR, 'Figure5_BCR_TCR_notitle.png'))
    plt.figure(figsize=(12, 10))
    plt.imshow(result)
    plt.title('Figure 5 — Title removed')
    plt.axis('off')
    plt.show()
    
else:
    print(f'❌ BCR/TCR figure not found at: {f8_path}')
    print('   Update F8_full path in Cell 1.')

In [ ]:
# Cell 6: Summary and output check

print('='*60)
print('FIGURE MERGING SUMMARY')
print('='*60)

expected_outputs = [
    'Figure1_merged.png',
    'Figure3_merged.png', 
    'Figure5_BCR_TCR_notitle.png',
]

for fname in expected_outputs:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        img = Image.open(fpath)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f'  ✅ {fname}: {img.size[0]}x{img.size[1]}px, {size_mb:.1f}MB')
    else:
        print(f'  ❌ {fname}: NOT FOUND')

print(f'\nOutput directory: {OUTPUT_DIR}')
print('\n--- Figures NOT requiring merging (just renumber) ---')
print('  Figure 2 = Old Figure 2 (pathway heatmap) — no change')
print('  Figure 4 = Old Figure 5 (correlation networks) — renumber only')
print('\n--- Supplementary Figures ---')
print('  Supp S3 = Old Main Figure 7 (six-layer schematic) — move to supp')
print('  Supp S8 = New (LAYN + JAK1 from dropped panels) — create separately')